# 03 预测与评估
**前置条件**：先运行 `01_data_processing.ipynb` → `02_model_training.ipynb`

---
## 📌 评估体系概览

### 为什么需要多角度评估？
单一指标无法全面反映模型质量，尤其在工业应用中：
- **RMSE**：惩罚大误差，适合质量异常检测场景
- **MAE**：鲁棒于异常值，反映日常运行精度
- **R²**：反映模型解释方差的比例（与基线的相对提升）
- **残差分析**：检测系统性偏差、异方差性
- **SHAP**：解释每个特征对单次预测的贡献，支持过程优化决策

## 1. 环境与数据加载

In [ ]:
import sys, warnings, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, yaml
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams.update({'figure.figsize': (13, 5), 'figure.dpi': 100})
sns.set_theme(style='whitegrid')
Path('../outputs/reports').mkdir(parents=True, exist_ok=True)
Path('../outputs/figures').mkdir(parents=True, exist_ok=True)

with open('../config.yaml', 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
print('环境初始化完成')

In [ ]:
X_train = np.load('../outputs/processed/X_train.npy')
X_test  = np.load('../outputs/processed/X_test.npy')
y_train = np.load('../outputs/processed/y_train.npy')
y_test  = np.load('../outputs/processed/y_test.npy')
selected_names = joblib.load('../outputs/processed/selected_names.pkl')

pls      = joblib.load('../outputs/models/pls.pkl')
xgb      = joblib.load('../outputs/models/xgboost.pkl')
lgbm     = joblib.load('../outputs/models/lightgbm.pkl')
stacking = joblib.load('../outputs/models/stacking.pkl')

print(f'测试集: X={X_test.shape}  y范围=[{y_test.min():.1f}, {y_test.max():.1f}] °C')
print(f'特征数: {len(selected_names)}')

## 2. 生成预测

In [ ]:
preds = {}
preds['PLS']      = pls.predict(X_test).ravel()
preds['XGBoost']  = xgb.predict(X_test)
preds['LightGBM'] = lgbm.predict(X_test)

base = stacking['base']; meta = stacking['meta']
meta_feat = np.column_stack([
    base['pls'].predict(X_test).ravel(),
    base['xgboost'].predict(X_test),
    base['lightgbm'].predict(X_test),
])
preds['Stacking'] = meta.predict(meta_feat)

# LSTM（如已训练）
LSTM_AVAIL = Path('../outputs/models/lstm_state.pt').exists()
if LSTM_AVAIL:
    import torch, torch.nn as nn
    lc = joblib.load('../outputs/models/lstm_config.pkl')
    class SoftSensorLSTM(nn.Module):
        def __init__(self, n_feat, hidden, n_layers):
            super().__init__()
            self.lstm = nn.LSTM(n_feat, hidden, n_layers, batch_first=True,
                                dropout=0.2 if n_layers > 1 else 0)
            self.fc = nn.Linear(hidden, 1)
        def forward(self, x):
            out, _ = self.lstm(x)
            return self.fc(out[:, -1, :]).squeeze(-1)
    net = SoftSensorLSTM(lc['n_feat'], lc['hidden'], lc['n_layers'])
    net.load_state_dict(torch.load('../outputs/models/lstm_state.pt', map_location='cpu'))
    net.eval()
    SEQ = lc['seq_len']
    seqs = np.array([X_test[i-SEQ:i] for i in range(SEQ, len(X_test))], dtype=np.float32)
    with torch.no_grad():
        lstm_pred = net(torch.tensor(seqs)).numpy()
    preds['LSTM'] = np.concatenate([np.full(SEQ, lstm_pred[0]), lstm_pred])
    print('LSTM预测已加载')

print(f'预测完成，模型: {list(preds.keys())}')

## 3. 评估指标汇总

> ### 📐 原理：各指标含义与选择
> 
> | 指标 | 公式 | 单位 | 含义 |
> |------|------|------|------|
> | **RMSE** | $\sqrt{\frac{1}{n}\sum(y_i - \hat{y}_i)^2}$ | °C | 对大误差惩罚更重（平方放大），适合评估极端偏差风险 |
> | **MAE**  | $\frac{1}{n}\sum|y_i - \hat{y}_i|$ | °C | 直觉上的平均误差，对异常值鲁棒 |
> | **R²**   | $1 - \frac{\sum(y_i-\hat{y}_i)^2}{\sum(y_i-\bar{y})^2}$ | 无 | 模型解释目标方差的比例，1=完美，0=与均值预测等价 |
> | **MAPE** | $\frac{100}{n}\sum\frac{|y_i-\hat{y}_i|}{|y_i|}$ | % | 相对误差，终馏点~170°C，MAPE<3%即为较好 |
> 
> **工业标准**：软测量RMSE < 3°C 可满足大多数炼化过程优化需求；R² > 0.9 为合格水平。

In [ ]:
def compute_metrics(y_true, y_pred):
    n = min(len(y_true), len(y_pred))
    yt, yp = y_true[:n], y_pred[:n]
    rmse = np.sqrt(mean_squared_error(yt, yp))
    mae  = mean_absolute_error(yt, yp)
    r2   = r2_score(yt, yp)
    nz   = yt != 0
    mape = np.mean(np.abs((yt[nz]-yp[nz])/yt[nz]))*100 if nz.any() else np.nan
    return {'RMSE(°C)': round(rmse,3), 'MAE(°C)': round(mae,3),
            'R²': round(r2,4), 'MAPE(%)': round(mape,3)}

metrics_all = {name: compute_metrics(y_test, pred) for name, pred in preds.items()}
metrics_df  = pd.DataFrame(metrics_all).T

print('=' * 60)
print('               测试集评估结果')
print('=' * 60)
print(metrics_df.to_string())
print(f'\n最优RMSE: {metrics_df["RMSE(°C)"].idxmin()} = {metrics_df["RMSE(°C)"].min():.3f}°C')
print(f'最优R²:   {metrics_df["R²"].idxmax()} = {metrics_df["R²"].max():.4f}')

with open('../outputs/reports/evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_all, f, ensure_ascii=False, indent=2)
metrics_df.to_csv('../outputs/reports/evaluation_metrics.csv')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['#4878CF', '#6ACC65', '#D65F5F', '#B47CC7', '#C4AD66']
for ax, metric in zip(axes, ['RMSE(°C)', 'MAE(°C)', 'R²']):
    vals = metrics_df[metric]
    bars = ax.bar(vals.index, vals.values, color=colors[:len(vals)], edgecolor='white', linewidth=0.8)
    ax.set_title(f'测试集 {metric}', fontsize=11)
    ax.tick_params(axis='x', rotation=30)
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/figures/metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 预测曲线 vs 实测值（时序对比）

In [ ]:
n_models = len(preds)
fig, axes = plt.subplots(n_models, 1, figsize=(14, 4*n_models), sharex=True)
if n_models == 1: axes = [axes]

for ax, (name, pred) in zip(axes, preds.items()):
    n = min(len(y_test), len(pred))
    ax.plot(y_test[:n], color='steelblue', linewidth=1.5, label='实测值', alpha=0.9)
    ax.plot(pred[:n],   color='tomato',    linewidth=1.5, label='预测值', linestyle='--', alpha=0.85)
    m = metrics_all[name]
    ax.set_title(f'{name}  |  R²={m["R²"]:.3f}  RMSE={m["RMSE(°C)"]:.2f}°C  MAE={m["MAE(°C)"]:.2f}°C')
    ax.set_ylabel('终馏点 (°C)'); ax.legend(loc='upper right', fontsize=9); ax.grid(True, alpha=0.4)

axes[-1].set_xlabel('测试集样本序号')
plt.tight_layout()
plt.savefig('../outputs/figures/prediction_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. 散点图（预测 vs 实测）

In [ ]:
n_cols = min(4, n_models)
n_rows = (n_models + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.5*n_cols, 5*n_rows), squeeze=False)
axes_flat = axes.flatten()

for ax, (name, pred) in zip(axes_flat, preds.items()):
    n = min(len(y_test), len(pred))
    yt, yp = y_test[:n], pred[:n]
    ax.scatter(yt, yp, alpha=0.4, s=15, color='steelblue')
    lim = [min(yt.min(), yp.min())-2, max(yt.max(), yp.max())+2]
    ax.plot(lim, lim, 'r--', linewidth=1.2, label='理想线 y=x')
    m = metrics_all[name]
    ax.set_title(f'{name}\nR²={m["R²"]:.3f}  RMSE={m["RMSE(°C)"]:.2f}°C')
    ax.set_xlabel('实测值 (°C)'); ax.set_ylabel('预测值 (°C)')
    ax.legend(fontsize=8); ax.set_xlim(lim); ax.set_ylim(lim)

for ax in axes_flat[n_models:]: ax.set_visible(False)
plt.tight_layout()
plt.savefig('../outputs/figures/prediction_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. 残差分析

> ### 📐 原理：残差分析告诉我们什么？
> 
> **残差** = 实测值 - 预测值，理想状态应满足：
> 1. **均值 ≈ 0**：无系统性偏差（偏高或偏低）
> 2. **正态分布**：误差来自随机噪声而非未建模的系统规律
> 3. **同方差性**：残差在不同预测值范围内方差相同（若大预测值时误差变大 = 异方差）
> 4. **无自相关**：时序上不应出现残差的漂移或周期性
> 
> **诊断问题**：
> - 残差均值 ≠ 0 → 模型有偏差，可能漏掉某个关键特征
> - 残差 vs 预测值呈漏斗形 → 异方差，考虑对目标取log变换
> - 残差时序有趋势 → 模型未捕捉过程漂移，需要在线更新

In [ ]:
fig, axes = plt.subplots(2, n_models, figsize=(5*n_models, 9), squeeze=False)
for j, (name, pred) in enumerate(preds.items()):
    n = min(len(y_test), len(pred))
    res = y_test[:n] - pred[:n]

    axes[0, j].hist(res, bins=30, color='steelblue', edgecolor='white')
    axes[0, j].axvline(0, color='red', linestyle='--')
    axes[0, j].axvline(res.mean(), color='orange', linestyle='-', label=f'均值={res.mean():.2f}')
    axes[0, j].set_title(f'{name}\n残差分布  σ={res.std():.2f}°C')
    axes[0, j].set_xlabel('残差 (°C)'); axes[0, j].legend(fontsize=8)

    axes[1, j].scatter(pred[:n], res, alpha=0.4, s=12, color='steelblue')
    axes[1, j].axhline(0, color='red', linestyle='--')
    axes[1, j].axhline( 2*res.std(), color='orange', linestyle=':', label='±2σ')
    axes[1, j].axhline(-2*res.std(), color='orange', linestyle=':')
    axes[1, j].set_title(f'{name}\n残差 vs 预测值')
    axes[1, j].set_xlabel('预测值 (°C)'); axes[1, j].set_ylabel('残差 (°C)')
    axes[1, j].legend(fontsize=8)

plt.tight_layout()
plt.savefig('../outputs/figures/residual_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
res_stats = []
for name, pred in preds.items():
    n = min(len(y_test), len(pred))
    res = y_test[:n] - pred[:n]
    res_stats.append({
        'Model': name,
        '均值(°C)': round(res.mean(), 3),
        'σ(°C)': round(res.std(), 3),
        '|误差|≤2°C(%)': round((np.abs(res)<=2).mean()*100, 1),
        '|误差|≤5°C(%)': round((np.abs(res)<=5).mean()*100, 1),
        'P95误差(°C)': round(np.percentile(np.abs(res), 95), 2)
    })
print('残差统计（工业评估关键指标）:')
print(pd.DataFrame(res_stats).set_index('Model').to_string())

## 7. SHAP 特征重要度分析

> ### 📐 原理：SHAP（SHapley Additive exPlanations）
> 
> **来源**：博弈论中的 Shapley 值 —— 公平分配多个参与者的合作收益
> 
> **计算**：对特征 $j$，穷举所有可能的特征子集 $S$，
> 计算加入特征 $j$ 前后模型预测的边际贡献并取期望：
> $$\phi_j = \sum_{S \subseteq F \setminus \{j\}} \frac{|S|!(|F|-|S|-1)!}{|F|!} [v(S \cup \{j\}) - v(S)]$$
> 
> **优势（对比传统特征重要度）**：
> - 每个样本都有独立SHAP值 → **局部可解释性**（为什么这次预测高/低）
> - 满足效率性、对称性、虚拟性公理 → **数学上唯一正确的特征归因**
> - 不受高基数特征偏倚（传统impurity-based重要度会高估高基数特征）
> 
> **图表解读**：
> - **蜂群图**：每个点=一个样本，横轴=SHAP值（正=推高预测，负=拉低），颜色=特征值大小
> - **Bar图**：|SHAP值|的全样本均值 = 全局特征重要度

In [ ]:
try:
    import shap
    short_names = [n.replace('CDU2.', '') for n in selected_names]
    explainer   = shap.TreeExplainer(xgb)
    sample      = X_test[:min(300, len(X_test))]
    shap_values = explainer.shap_values(sample)

    # 蜂群图（最信息量丰富）
    plt.figure(figsize=(9, 7))
    shap.summary_plot(shap_values, sample, feature_names=short_names, show=False)
    plt.title('XGBoost SHAP蜂群图\n点=样本, 横轴=对预测贡献, 颜色=特征值(红高蓝低)')
    plt.tight_layout()
    plt.savefig('../outputs/figures/shap_summary.png', dpi=150, bbox_inches='tight')
    plt.show()

    # 全局重要度柱状图
    plt.figure(figsize=(9, 6))
    shap.summary_plot(shap_values, sample, feature_names=short_names, plot_type='bar', show=False)
    plt.title('XGBoost SHAP全局重要度（|SHAP值|均值）')
    plt.tight_layout()
    plt.savefig('../outputs/figures/shap_bar.png', dpi=150, bbox_inches='tight')
    plt.show()

except ImportError:
    print('未安装shap: pip install shap')

In [ ]:
# SHAP依赖图：最重要的3个特征与预测的关系
try:
    top3 = np.argsort(np.abs(shap_values).mean(0))[::-1][:3]
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, feat_i in zip(axes, top3):
        shap.dependence_plot(feat_i, shap_values, sample,
                             feature_names=short_names, ax=ax, show=False)
        ax.set_title(f'SHAP依赖: {short_names[feat_i]}')
    plt.suptitle('SHAP依赖图：特征值变化如何影响对终馏点的预测贡献', y=1.02)
    plt.tight_layout()
    plt.savefig('../outputs/figures/shap_dependence.png', dpi=150, bbox_inches='tight')
    plt.show()
except Exception as e:
    print(f'SHAP依赖图跳过: {e}')

## 8. 大误差案例分析

> ### 📐 原理：为什么分析大误差样本？
> 
> 大误差样本往往揭示模型的系统性盲区：
> - **特定操作区间**：切换产品牌号时、开停工过渡态时误差偏大
> - **传感器异常**：某关键仪表短暂异常但未被坏值检测捕捉
> - **未建模因素**：换热器结垢、塔内件堵塞等慢变故障
> 
> 通过分析大误差样本的特征值分布，可以指导：
> 1. 是否需要增加新特征（如设备状态指标）
> 2. 是否需要分工况建立多个模型
> 3. 是否需要定期在线更新模型（应对过程漂移）

In [ ]:
best_model = metrics_df['RMSE(°C)'].idxmin()
best_pred  = preds[best_model]
n = min(len(y_test), len(best_pred))
res = y_test[:n] - best_pred[:n]

error_df = pd.DataFrame({'实测值': y_test[:n], '预测值': best_pred[:n],
                         '残差': res, '|残差|': np.abs(res)})
thresh_90 = np.percentile(np.abs(res), 90)
large_err = error_df[error_df['|残差|'] > thresh_90]

print(f'最优模型: {best_model}')
print(f'P90误差阈值: {thresh_90:.2f}°C  |  超出样本数: {len(large_err)}')
print('\nTop-10 最大误差样本:')
print(large_err.nlargest(10, '|残差|').round(2).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].scatter(error_df['实测值'], error_df['|残差|'], alpha=0.4, s=15, color='steelblue')
axes[0].axhline(thresh_90, color='red', linestyle='--', label=f'P90={thresh_90:.2f}°C')
axes[0].set_title(f'{best_model}: 误差 vs 实测值')
axes[0].set_xlabel('实测终馏点 (°C)'); axes[0].set_ylabel('|残差| (°C)')
axes[0].legend()

window = max(10, n // 20)
rolling_rmse = pd.Series(res**2).rolling(window, min_periods=1).mean().apply(np.sqrt)
axes[1].plot(rolling_rmse, color='tomato', linewidth=1.5)
axes[1].axhline(rolling_rmse.mean(), color='steelblue', linestyle='--',
                label=f'均值={rolling_rmse.mean():.2f}°C')
axes[1].set_title(f'{best_model}: 滚动RMSE（窗口={window}）\n趋势上升 → 过程漂移，需重新训练')
axes[1].set_xlabel('样本序号'); axes[1].set_ylabel('RMSE (°C)'); axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/error_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. 汇总

In [ ]:
print('=' * 60)
print('           石脑油终馏点软测量 —— 最终评估汇总')
print('=' * 60)
print(metrics_df.to_string())
print('\n📁 生成文件:')
for p in sorted(Path('../outputs').rglob('*')):
    if p.is_file():
        size = p.stat().st_size
        val  = size/1024 if size < 1024*1024 else size/1024/1024
        unit = 'KB' if size < 1024*1024 else 'MB'
        print(f'  {str(p.relative_to("../"))}  ({val:.1f} {unit})')